# Aggregator — Ranking Final Seleksi SLM
Membaca semua file hasil evaluasi per-model dan menghasilkan ranking final.

## Section 1 — Load Semua File Hasil

In [1]:
import json, glob, os
import pandas as pd

RESULT_DIR = "/kaggle/input/datasets/muhammadnabilul/slm-eval-result"

summary_files      = sorted(glob.glob(os.path.join(RESULT_DIR, "**", "summary_*.json"), recursive=True))
mmlu_subj_files    = sorted(glob.glob(os.path.join(RESULT_DIR, "**", "breakdown_mmlu_subject_*.csv"), recursive=True))
career_group_files = sorted(glob.glob(os.path.join(RESULT_DIR, "**", "breakdown_career_group_*.csv"), recursive=True))

model_names = [os.path.basename(f).replace("summary_","").replace(".json","") for f in summary_files]
print(f"Model ditemukan ({len(model_names)}):") 
for m in model_names:
    print(" -", m)

summaries = []
for f in summary_files:
    with open(f) as fp:
        summaries.append(json.load(fp))

df_mmlu_subj_all    = pd.concat([pd.read_csv(f) for f in mmlu_subj_files],    ignore_index=True)
df_career_group_all = pd.concat([pd.read_csv(f) for f in career_group_files], ignore_index=True)

print("\nSemua file berhasil dimuat.")

Model ditemukan (12):
 - Gemma-2-2B
 - Llama-3.1-8B
 - Llama3.2-1B
 - Llama3.2-3B
 - Mistral-7B
 - Phi-3.5-mini
 - Qwen2.5-0.5B
 - Qwen2.5-1.5B
 - Qwen2.5-3B
 - Qwen2.5-7B
 - SmolLM2-135M
 - SmolLM2-360M

Semua file berhasil dimuat.


In [2]:
for f in mmlu_subj_files:
    print(f)

/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Gemma-2-2B/breakdown_mmlu_subject_Gemma-2-2B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Llama-3.1-8B/breakdown_mmlu_subject_Llama-3.1-8B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Llama3.2-1B/breakdown_mmlu_subject_Llama3.2-1B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Llama3.2-3B/breakdown_mmlu_subject_Llama3.2-3B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Mistral-7B/breakdown_mmlu_subject_Mistral-7B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Phi-3.5-mini/breakdown_mmlu_subject_Phi-3.5-mini.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Qwen2.5-0.5B/breakdown_mmlu_subject_Qwen2.5-0.5B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Qwen2.5-1.5B/breakdown_mmlu_subject_Qwen2.5-1.5B.csv
/kaggle/input/datasets/muhammadnabilul/slm-eval-result/Qwen2.5-3B/breakdown_mmlu_subject_Qwen2.5-3B.csv
/kaggle/input/datasets/muhammadnabilul/slm-e

In [3]:
df_mmlu_subj_all 

,subject,accuracy,n_soal,model
0,IPS,66.61,596,Gemma-2-2B
1,IPA,55.95,967,Gemma-2-2B
2,Kesenian,53.68,598,Gemma-2-2B
3,Agama Kristen,53.03,198,Gemma-2-2B
4,PPKN,52.44,696,Gemma-2-2B
...,...,...,...,...
295,Sosiologi,22.52,493,SmolLM2-360M
296,Kimia,21.70,682,SmolLM2-360M
297,Geografi,21.36,487,SmolLM2-360M
298,Fisika,20.12,492,SmolLM2-360M


In [4]:
df_career_group_all

,profession_group,accuracy,n_soal,model
0,Tourism and Hospitality,52.86,874,Gemma-2-2B
1,Creative and Design,52.77,1103,Gemma-2-2B
2,Education and Training,49.20,878,Gemma-2-2B
3,Insurance and Financial,48.16,2388,Gemma-2-2B
4,Legal and Law Enforcement,41.04,865,Gemma-2-2B
...,...,...,...,...
67,Tourism and Hospitality,25.40,874,SmolLM2-360M
68,Healthcare and Medical,24.36,2660,SmolLM2-360M
69,Legal and Law Enforcement,23.93,865,SmolLM2-360M
70,Creative and Design,23.39,1103,SmolLM2-360M


## Section 2 — Ranking Summary (diurutkan Final Score tertinggi)

In [5]:
df_summary = pd.DataFrame(summaries)
df_summary = df_summary.sort_values("final_score_50_50", ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = "rank"

print("=== RANKING FINAL ===")
print(df_summary[["model_name", "indommlu_acc", "indocareer_acc", "final_score_50_50", "indommlu_n", "indocareer_n"]].to_string())
print(f"\nTOP 2 → Fine-Tuning: {df_summary.iloc[0]['model_name']} & {df_summary.iloc[1]['model_name']}")

df_summary.to_csv("ranking_summary.csv", index=True)

=== RANKING FINAL ===
        model_name  indommlu_acc  indocareer_acc  final_score_50_50  indommlu_n  indocareer_n
rank                                                                                         
1     Llama-3.1-8B       48.5576         50.3992            49.4784       14906          8768
2      Llama3.2-3B       44.8880         45.5178            45.2029       14906          8768
3       Gemma-2-2B       43.9890         43.7386            43.8638       14906          8768
4       Mistral-7B       43.2980         44.0237            43.6609       14906          8768
5       Qwen2.5-7B       40.2590         41.7085            40.9837       14906          8768
6     Phi-3.5-mini       40.1114         41.0128            40.5621       14906          8768
7      Llama3.2-1B       31.8328         31.4667            31.6498       14906          8768
8       Qwen2.5-3B       29.9007         29.6419            29.7713       14906          8768
9     Qwen2.5-1.5B       26.4390      

## Section 3 — Agregat Breakdown IndoMMLU per Subject

In [6]:
df_mmlu_subj_all = df_mmlu_subj_all.drop_duplicates(subset=["model", "subject"], keep="first")

df_mmlu_subj_pivot = df_mmlu_subj_all.pivot_table(
    index="model", columns="subject", values="accuracy"
).reset_index()

df_mmlu_subj_pivot = df_mmlu_subj_pivot.merge(
    df_summary[["model_name", "indommlu_acc"]].rename(columns={"model_name": "model"}),
    on="model", how="left"
).sort_values("indommlu_acc", ascending=False).reset_index(drop=True)

df_mmlu_subj_pivot.index += 1
df_mmlu_subj_pivot.index.name = "rank"

print(df_mmlu_subj_pivot.to_string())
df_mmlu_subj_pivot.to_csv("breakdown_mmlu_subject_all.csv", index=True)

             model  Agama Hindu  Agama Islam  Agama Kristen  Bahasa Bali  Bahasa Banjar  Bahasa Dayak Ngaju  Bahasa Indonesia  Bahasa Jawa  Bahasa Lampung  Bahasa Madura  Bahasa Makassar  Bahasa Sunda  Biologi  Budaya Alam Minangkabau  Ekonomi  Fisika  Geografi    IPA    IPS  Kesenian  Kimia   PPKN  Penjaskes  Sejarah  Sosiologi  indommlu_acc
rank                                                                                                                                                                                                                                                                                                                                                    
1     Llama-3.1-8B        47.62        60.57          58.08        30.56          30.50               27.36             54.24        34.88           34.03          23.97            28.42         37.44    49.05                    37.24    48.04   40.65     45.38  61.53  75.00     58.36  32.26  61.06      45.52

In [7]:
df_mmlu_subj_pivot

,model,Agama Hindu,Agama Islam,Agama Kristen,Bahasa Bali,Bahasa Banjar,Bahasa Dayak Ngaju,Bahasa Indonesia,Bahasa Jawa,Bahasa Lampung,...,Geografi,IPA,IPS,Kesenian,Kimia,PPKN,Penjaskes,Sejarah,Sosiologi,indommlu_acc
rank,,,,,,,,,,,,,,,,,,,,,
1,Llama-3.1-8B,47.62,60.57,58.08,30.56,30.50,27.36,54.24,34.88,34.03,...,45.38,61.53,75.00,58.36,32.26,61.06,45.52,46.46,51.32,48.5576
2,Llama3.2-3B,37.41,50.57,54.55,26.92,34.75,30.19,52.68,32.86,31.25,...,40.04,55.64,65.10,54.85,30.06,51.72,46.90,41.21,47.87,44.8880
3,Gemma-2-2B,41.50,51.14,53.03,29.91,36.17,25.47,50.34,30.84,31.25,...,40.86,55.95,66.61,53.68,27.57,52.44,52.41,38.59,44.83,43.9890
4,Mistral-7B,51.02,51.00,47.98,31.84,39.01,24.53,48.22,30.94,29.86,...,39.01,50.36,61.24,50.50,32.26,49.57,47.59,44.65,44.42,43.2980
5,Qwen2.5-7B,40.14,52.71,53.03,28.21,30.50,35.85,42.27,29.32,32.64,...,39.43,50.26,57.55,48.66,27.27,46.12,42.07,36.57,39.76,40.2590
6,Phi-3.5-mini,42.18,44.29,45.45,30.77,39.01,29.25,45.64,32.96,34.03,...,37.78,47.67,57.38,40.13,29.62,40.95,36.55,35.76,41.78,40.1114
7,Llama3.2-1B,35.37,35.57,29.29,27.14,33.33,31.13,34.77,29.32,26.39,...,29.77,36.09,41.44,31.77,25.66,33.91,27.59,28.69,27.79,31.8328
8,Qwen2.5-3B,29.93,30.57,32.83,30.98,23.40,18.87,31.56,27.81,31.25,...,24.02,34.95,39.77,34.11,24.19,33.48,32.41,24.44,27.99,29.9007
9,Qwen2.5-1.5B,28.57,27.86,29.29,30.13,26.95,28.30,29.47,23.36,30.56,...,19.10,27.51,31.54,24.75,22.43,28.02,23.45,26.06,24.75,26.4390


## Section 4 — Agregat Breakdown IndoCareer per Group 

In [8]:
df_career_group_all = df_career_group_all.drop_duplicates(subset=["model", "profession_group"], keep="first")

df_career_group_pivot = df_career_group_all.pivot_table(
    index="model", columns="profession_group", values="accuracy"
).reset_index()

df_career_group_pivot = df_career_group_pivot.merge(
    df_summary[["model_name", "indocareer_acc"]].rename(columns={"model_name": "model"}),
    on="model", how="left"
).sort_values("indocareer_acc", ascending=False).reset_index(drop=True)

df_career_group_pivot.index += 1
df_career_group_pivot.index.name = "rank"

print(df_career_group_pivot.to_string())
df_career_group_pivot.to_csv("breakdown_mmlu_subject_all.csv", index=True)

             model  Creative and Design  Education and Training  Healthcare and Medical  Insurance and Financial  Legal and Law Enforcement  Tourism and Hospitality  indocareer_acc
rank                                                                                                                                                                                
1     Llama-3.1-8B                61.38                   51.71                   43.05                    51.38                      46.01                    59.27         50.3992
2      Llama3.2-3B                54.49                   49.32                   38.01                    47.57                      41.04                    52.06         45.5178
3       Mistral-7B                53.58                   48.18                   33.87                    46.52                      41.85                    54.00         44.0237
4       Gemma-2-2B                52.77                   49.20                   32.11        

In [9]:
df_career_group_pivot

,model,Creative and Design,Education and Training,Healthcare and Medical,Insurance and Financial,Legal and Law Enforcement,Tourism and Hospitality,indocareer_acc
rank,,,,,,,,
1,Llama-3.1-8B,61.38,51.71,43.05,51.38,46.01,59.27,50.3992
2,Llama3.2-3B,54.49,49.32,38.01,47.57,41.04,52.06,45.5178
3,Mistral-7B,53.58,48.18,33.87,46.52,41.85,54.00,44.0237
4,Gemma-2-2B,52.77,49.20,32.11,48.16,41.04,52.86,43.7386
5,Qwen2.5-7B,48.50,43.85,34.96,45.14,40.81,43.02,41.7085
6,Phi-3.5-mini,50.68,42.60,34.44,43.22,37.57,44.62,41.0128
7,Llama3.2-1B,35.81,33.71,27.03,33.00,31.21,33.30,31.4667
8,Qwen2.5-3B,33.00,29.61,26.47,31.20,27.05,33.41,29.6419
9,Qwen2.5-1.5B,26.75,26.20,23.35,28.02,25.90,22.88,25.5360


## Section 5 — Tabel Final Komprehensif

In [10]:
df_mmlu_subj_pivot_clean   = df_mmlu_subj_pivot.drop_duplicates(subset=["model"], keep="first")
df_career_group_pivot_clean = df_career_group_pivot.drop_duplicates(subset=["model"], keep="first")

df_final = df_summary[["model_name", "indommlu_acc", "indocareer_acc", "final_score_50_50"]].copy()
df_final = df_final.rename(columns={"model_name": "model"})

mmlu_subj_cols    = [c for c in df_mmlu_subj_pivot_clean.columns    if c not in ["model", "indommlu_acc"]]
career_group_cols = [c for c in df_career_group_pivot_clean.columns if c not in ["model", "indocareer_acc"]]

df_final = df_final.merge(
    df_mmlu_subj_pivot_clean[["model"] + mmlu_subj_cols].rename(columns={c: f"mmlu_{c}" for c in mmlu_subj_cols}),
    on="model", how="left"
)
df_final = df_final.merge(
    df_career_group_pivot_clean[["model"] + career_group_cols].rename(columns={c: f"career_{c}" for c in career_group_cols}),
    on="model", how="left"
)

df_final = df_final.sort_values("final_score_50_50", ascending=False).reset_index(drop=True)
df_final.index += 1
df_final.index.name = "rank"

print("=== TABEL FINAL KOMPREHENSIF ===")
print(df_final.to_string())
df_final.to_csv("ranking_final_komprehensif.csv", index=True)
print("\nSemua hasil disimpan.")

=== TABEL FINAL KOMPREHENSIF ===
             model  indommlu_acc  indocareer_acc  final_score_50_50  mmlu_Agama Hindu  mmlu_Agama Islam  mmlu_Agama Kristen  mmlu_Bahasa Bali  mmlu_Bahasa Banjar  mmlu_Bahasa Dayak Ngaju  mmlu_Bahasa Indonesia  mmlu_Bahasa Jawa  mmlu_Bahasa Lampung  mmlu_Bahasa Madura  mmlu_Bahasa Makassar  mmlu_Bahasa Sunda  mmlu_Biologi  mmlu_Budaya Alam Minangkabau  mmlu_Ekonomi  mmlu_Fisika  mmlu_Geografi  mmlu_IPA  mmlu_IPS  mmlu_Kesenian  mmlu_Kimia  mmlu_PPKN  mmlu_Penjaskes  mmlu_Sejarah  mmlu_Sosiologi  career_Creative and Design  career_Education and Training  career_Healthcare and Medical  career_Insurance and Financial  career_Legal and Law Enforcement  career_Tourism and Hospitality
rank                                                                                                                                                                                                                                                                                   

In [11]:
df_final

,model,indommlu_acc,indocareer_acc,final_score_50_50,mmlu_Agama Hindu,mmlu_Agama Islam,mmlu_Agama Kristen,mmlu_Bahasa Bali,mmlu_Bahasa Banjar,mmlu_Bahasa Dayak Ngaju,...,mmlu_PPKN,mmlu_Penjaskes,mmlu_Sejarah,mmlu_Sosiologi,career_Creative and Design,career_Education and Training,career_Healthcare and Medical,career_Insurance and Financial,career_Legal and Law Enforcement,career_Tourism and Hospitality
rank,,,,,,,,,,,,,,,,,,,,,
1,Llama-3.1-8B,48.5576,50.3992,49.4784,47.62,60.57,58.08,30.56,30.50,27.36,...,61.06,45.52,46.46,51.32,61.38,51.71,43.05,51.38,46.01,59.27
2,Llama3.2-3B,44.8880,45.5178,45.2029,37.41,50.57,54.55,26.92,34.75,30.19,...,51.72,46.90,41.21,47.87,54.49,49.32,38.01,47.57,41.04,52.06
3,Gemma-2-2B,43.9890,43.7386,43.8638,41.50,51.14,53.03,29.91,36.17,25.47,...,52.44,52.41,38.59,44.83,52.77,49.20,32.11,48.16,41.04,52.86
4,Mistral-7B,43.2980,44.0237,43.6609,51.02,51.00,47.98,31.84,39.01,24.53,...,49.57,47.59,44.65,44.42,53.58,48.18,33.87,46.52,41.85,54.00
5,Qwen2.5-7B,40.2590,41.7085,40.9837,40.14,52.71,53.03,28.21,30.50,35.85,...,46.12,42.07,36.57,39.76,48.50,43.85,34.96,45.14,40.81,43.02
6,Phi-3.5-mini,40.1114,41.0128,40.5621,42.18,44.29,45.45,30.77,39.01,29.25,...,40.95,36.55,35.76,41.78,50.68,42.60,34.44,43.22,37.57,44.62
7,Llama3.2-1B,31.8328,31.4667,31.6498,35.37,35.57,29.29,27.14,33.33,31.13,...,33.91,27.59,28.69,27.79,35.81,33.71,27.03,33.00,31.21,33.30
8,Qwen2.5-3B,29.9007,29.6419,29.7713,29.93,30.57,32.83,30.98,23.40,18.87,...,33.48,32.41,24.44,27.99,33.00,29.61,26.47,31.20,27.05,33.41
9,Qwen2.5-1.5B,26.4390,25.5360,25.9875,28.57,27.86,29.29,30.13,26.95,28.30,...,28.02,23.45,26.06,24.75,26.75,26.20,23.35,28.02,25.90,22.88
